In [1]:
import requests

# ----------------------------
# TOOL
# ----------------------------
def weather_tool(city: str):
    url = f"https://wttr.in/{city}?format=j1"
    response = requests.get(url, timeout=10)
    response.raise_for_status()
    data = response.json()

    current = data["current_condition"][0]

    return {
        "city": city,
        "temperature_c": current["temp_C"],
        "description": current["weatherDesc"][0]["value"],
        "humidity": current["humidity"]
    }


# ----------------------------
# AGENT
# ----------------------------
class TravelAgent:
    def think(self, user_query: str):
        """
        Decide what to do based on user query.
        """
        query = user_query.lower()

        if "weather" in query or "umbrella" in query or "rain" in query:
            return {
                "tool_needed": "weather_tool",
                "city": "Chennai"
            }

        return {
            "tool_needed": None,
            "city": None
        }

    def analyze_weather(self, weather_data: dict):
        """
        Interpret tool result and produce final decision.
        """
        desc = weather_data["description"].lower()
        temp = int(weather_data["temperature_c"])

        if "rain" in desc or "drizzle" in desc or "thunder" in desc:
            return f"Yes, carry an umbrella. Current weather in {weather_data['city']} is {weather_data['description']}."
        elif temp >= 38:
            return f"No umbrella needed, but it is very hot in {weather_data['city']} ({temp}°C). Carry water."
        else:
            return f"Umbrella may not be necessary now. Current weather in {weather_data['city']} is {weather_data['description']} with {temp}°C."

    def respond(self, user_query: str):
        """
        Full agent flow:
        1. Think
        2. Decide tool
        3. Call tool
        4. Analyze
        5. Respond
        """
        print("User Query:", user_query)

        decision = self.think(user_query)
        print("Agent Decision:", decision)

        if decision["tool_needed"] == "weather_tool":
            print("Agent is calling weather tool...")
            weather_data = weather_tool(decision["city"])
            print("Tool Output:", weather_data)

            final_answer = self.analyze_weather(weather_data)
            return final_answer

        return "I do not need any external tool for this question."


# ----------------------------
# RUN
# ----------------------------
agent = TravelAgent()

query = "Should I carry an umbrella in Chennai today?"
answer = agent.respond(query)

print("\nFinal Answer:")
print(answer)

User Query: Should I carry an umbrella in Chennai today?
Agent Decision: {'tool_needed': 'weather_tool', 'city': 'Chennai'}
Agent is calling weather tool...
Tool Output: {'city': 'Chennai', 'temperature_c': '32', 'description': 'Sunny', 'humidity': '67'}

Final Answer:
Umbrella may not be necessary now. Current weather in Chennai is Sunny with 32°C.
